## Training neural networks

In [4]:
# =============================================
# NOTEBOOK 1: TRAIN MODEL (CIFAR-like datasets)
# =============================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader

# ----------------------
# CONFIGURATION
# ----------------------
ARCH = "resnet18"  # choices: resnet18, resnet34, vgg11, mobilenet_v2
DATASET = "CIFAR10"  # choices: CIFAR10, CIFAR100
BATCH_SIZE = 128
EPOCHS = 10
LR = 0.001
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"

# ----------------------
# DATA
# ----------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

if DATASET == "CIFAR10":
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    num_classes = 10

elif DATASET == "CIFAR100":
    trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
    num_classes = 100

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

# ----------------------
# MODEL SELECTION
# ----------------------
def get_model(arch, num_classes):
    if arch == "resnet18":
        model = models.resnet18()
        for layer in model.children():
            print("Name layer : ", layer)
            if isinstance(layer, nn.MaxPool2d):
                print("Modifying MaxPool2d layer")
                layer.kernel_size = 2
                layer.stride = 2
                layer.padding = 0
        model.maxpool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch == "resnet34":
        model = models.resnet34()
        model.maxpool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif arch == "vgg11":
        model = models.vgg11()
        model.classifier[6] = nn.Linear(4096, num_classes)

    elif arch == "mobilenet_v2":
        model = models.mobilenet_v2()
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError("Unknown architecture")

    return model

model = get_model(ARCH, num_classes).to(DEVICE)

# ----------------------
# TRAINING
# ----------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(trainloader):.4f}")

# ----------------------
# SAVE MODEL
# ----------------------
PATH = f"/share/homes/boyerma/robustesse_ood/models/{ARCH}_{DATASET}.pth"
torch.save(model.state_dict(), PATH)
print(f"Model saved at {PATH}")





Name layer :  Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
Name layer :  BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
Name layer :  ReLU(inplace=True)
Name layer :  MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
Modifying MaxPool2d layer
Name layer :  Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (1): BasicBlock(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inpla